# Trace an agent run and evaluate its answer

This short application lab records selected AgentScope events from one tool-using run, then checks the final answer against a small, visible rubric. Lesson 04 explains the individual event names; this notebook uses them to accept or reject a run.

## Trace versus evaluation

A trace records **what happened** during the run. An evaluation checks whether the answer meets stated criteria. Accept the run only when both support it; neither replaces the other.

## Import the lesson components

The next cell imports the AgentScope classes, the local-model configuration helpers, and `Counter`, which will summarize the saved trace events.

In [ ]:
from collections import Counter
import os

from dotenv import load_dotenv
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel
from agentscope.tool import FunctionTool, Toolkit

## Define the fixed practice record

The next cell creates one read-only tool backed by fictional local data. Its known result gives the evaluator a clear standard for judging the final answer.

In [ ]:
def get_ip_details(ip_address: str) -> dict[str, str]:
    """Return a record from the fixed practice list.

    Args:
        ip_address: An internet address to look up.
    """
    # This fixed data makes the expected evidence clear and repeatable.
    records = {
        "192.0.2.44": {
            "local_result": "suspicious",
            "details": "A person should look into this address in the practice scenario; this is not proof of malicious activity.",
        },
    }
    return records.get(ip_address, {
        "local_result": "no record",
        "details": "The practice list has no details for this address.",
    })

# FunctionTool supplies a schema the model can use to request this read-only function.
toolkit = Toolkit(tools=[FunctionTool(get_ip_details, is_read_only=True)])

## Configure the model and agent

The next cell loads the local model settings and creates an agent whose instructions require the practice tool and prohibit an unsupported conclusion.

In [ ]:
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=180),
)

agent = Agent(
    name="evidence_assistant",
    system_prompt=(
        "For every internet-address question, call get_ip_details before answering. "
        "State the address and its local result. "
        "If the local result is suspicious, say it needs review and is not proof of malicious activity."
    ),
    model=model,
    toolkit=toolkit,
    react_config=ReActConfig(max_iters=3),
)

## Record a small execution trace

`reply_stream` yields many events. This cell stores the names of the major events in a list, then prints a compact count. The saved trace is evidence of the run’s behavior, not evidence that the final conclusion is correct.

In [ ]:
question = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text="What does the practice list say about 192.0.2.44?")],
)

# Lesson 04 explains these event types. Here we save only the high-level
# events needed to verify that this run followed the expected path.
tracked_events = {
    "ModelCallStartEvent": "Model request starts",
    "ToolCallStartEvent": "Tool requested",
    "ToolResultStartEvent": "Tool starts",
    "ToolResultEndEvent": "Tool result is ready",
    "ReplyEndEvent": "Agent finished its answer",
}
trace_events = []
final_response = None

# The stream yields event objects while work happens and one final Msg.
async for item in agent.reply_stream(question, yield_final_msg=True):
    if isinstance(item, Msg):
        final_response = item
    else:
        event_name = item.__class__.__name__
        if event_name in tracked_events:
            trace_events.append(event_name)

# A final message is required before we can evaluate the answer.
if final_response is None:
    raise RuntimeError("The agent did not return a final response.")

trace_counts = Counter(trace_events)
print("TRACE SUMMARY:")
for event_name, label in tracked_events.items():
    print(f"{label}: {trace_counts[event_name]}")

## Extract the final answer

The next cell turns the agent’s final message into plain text. That text is displayed for review and then passed to the evaluation checklist.

In [ ]:
# Extract text from the final Msg so the evaluator can examine it.
final_answer = "".join(
    block.text for block in final_response.content if isinstance(block, TextBlock)
)

print("FINAL ANSWER:")
print(final_answer)

## Evaluate the final answer

This is a small deterministic evaluator, not an LLM judge. Each check is visible and tied to the known practice record. In a real course, you could replace or expand this with a scored rubric and human review.

In [ ]:
def evaluate_answer(answer: str) -> list[tuple[str, bool]]:
    """Check whether an answer meets the three practice-case criteria."""
    normalized = answer.lower()
    return [
        ("states the correct address", "192.0.2.44" in normalized),
        ("reports the suspicious local result", "suspicious" in normalized),
        ("avoids an unsupported claim", "not proof of malicious activity" in normalized),
    ]

evaluation = evaluate_answer(final_answer)

print("EVALUATION:")
for criterion, passed in evaluation:
    status = "PASS" if passed else "CHECK"
    print(f"{status}: {criterion}")

if all(passed for _, passed in evaluation):
    print("\nThe answer meets this lesson's visible checklist.")
else:
    print("\nReview the final answer and the trace before accepting this result.")

## Checkpoint

Temporarily remove the final sentence from the agent's system prompt: `If the local result is suspicious, ...`. Rerun from **Configure the agent** through **Evaluate the final answer**. If the answer overclaims, the last evaluation criterion should show `CHECK`. Restore the original prompt when finished.